# SimCLR — Implementation / 구현

**Paper**: Chen, T., Kornblith, S., Norouzi, M., Hinton, G. (2020). *A Simple Framework for Contrastive Learning of Visual Representations*. ICML 2020. arXiv:2002.05709.

이 노트북은 SimCLR의 핵심 구성요소를 PyTorch로 직접 구현하고, 합성 2D 클러스터 데이터에 대해 mini-SimCLR 데모를 수행합니다.

This notebook implements SimCLR's core components from scratch in PyTorch and runs a mini-SimCLR demo on synthetic 2D cluster data, then visualises the learned embeddings.

**Pipeline**:
1. **NT-Xent loss** — Normalized Temperature-scaled Cross-entropy from scratch
2. **Tiny encoder + projection head** — 2-layer MLP encoder $f$, 2-layer MLP projection head $g$
3. **Augmentation** — random Gaussian noise on 2D points to create positive pairs
4. **Training loop** — pretrain on synthetic clusters
5. **Visualization** — compare $h$ before training, $h$ after training, $z=g(h)$ (t-SNE)

**Kernel**: `study-with-ai`

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Part 1: NT-Xent Loss from Scratch / NT-Xent 손실 직접 구현

**한국어**
NT-Xent (Normalized Temperature-scaled Cross-Entropy) 손실은 SimCLR의 핵심입니다. 미니배치 $N$개 입력에 대해 두 개의 augmentation view가 만들어져 총 $2N$개 임베딩이 됩니다. positive pair $(i, j)$에 대해:

$$\ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k)/\tau)}$$

구현은 (1) $\ell_2$ 정규화, (2) 모든 쌍 cosine similarity 행렬, (3) 자기 자신 마스킹, (4) cross-entropy로 진행합니다.

**English**
NT-Xent is SimCLR's core loss. For a mini-batch of $N$ inputs, augmentation creates $2N$ embeddings. For positive pair $(i, j)$:

$$\ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k)/\tau)}$$

Implementation: (1) $\ell_2$ normalize, (2) full pairwise cosine similarity matrix, (3) mask self-similarities, (4) reduce via cross-entropy.

In [ ]:
class NTXentLoss(nn.Module):
    """Normalized Temperature-scaled Cross Entropy loss (SimCLR, Eq. 1).

    Given two batches of projections z_a, z_b each of shape (N, d), where row k of
    z_a and row k of z_b are two augmented views of the same image (positive pair),
    this module computes the average NT-Xent loss across all 2N positive pairs.

    Args:
        temperature: softmax temperature tau in Eq. (1). Typical 0.1-0.5.
    """

    def __init__(self, temperature: float = 0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_a: torch.Tensor, z_b: torch.Tensor) -> torch.Tensor:
        """Compute NT-Xent loss.

        Args:
            z_a: projection vectors from view a, shape (N, d).
            z_b: projection vectors from view b, shape (N, d).

        Returns:
            Scalar loss tensor (mean over 2N positive pairs).
        """
        n = z_a.size(0)
        # L2 normalize -> cosine similarity reduces to dot product
        z_a = F.normalize(z_a, dim=1)
        z_b = F.normalize(z_b, dim=1)
        # Stack to (2N, d). Row i and row i+N are positives.
        z = torch.cat([z_a, z_b], dim=0)
        # Pairwise cosine similarity (2N, 2N)
        sim = torch.mm(z, z.t()) / self.temperature
        # Mask diagonal (self-similarity) with -inf
        mask_self = torch.eye(2 * n, dtype=torch.bool, device=z.device)
        sim.masked_fill_(mask_self, float('-inf'))
        # Targets: for row i in [0, N), positive is at column i+N. For row i in [N, 2N),
        # positive is at column i-N.
        targets = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
        loss = F.cross_entropy(sim, targets)
        return loss


# Sanity check: random embeddings should yield loss near log(2N - 1)
criterion = NTXentLoss(temperature=0.5)
z_a_rand = torch.randn(8, 16)
z_b_rand = torch.randn(8, 16)
loss_rand = criterion(z_a_rand, z_b_rand)
expected = math.log(2 * 8 - 1)
print(f'Random NT-Xent loss: {loss_rand.item():.4f}')
print(f'Expected (log(2N-1)): {expected:.4f}')

# Sanity check: identical pairs (perfect agreement) should give very low loss
z_perfect = F.normalize(torch.randn(8, 16), dim=1)
loss_perfect = criterion(z_perfect, z_perfect.clone())
print(f'Perfect-agreement loss: {loss_perfect.item():.4f} (should be << random)')

## Part 2: Synthetic Dataset and Augmentation / 합성 데이터와 증강

**한국어**
이미지 대신 2D 클러스터 데이터를 사용해 SimCLR의 학습 동역학을 빠르게 시각화합니다. 4개의 Gaussian 클러스터에서 점을 샘플링하고, augmentation으로 작은 등방성 노이즈를 더해 positive pair를 만듭니다 — 같은 점에서 두 번 노이즈를 더한 결과는 "같은 의미(클러스터)"를 공유하지만 좌표는 다릅니다. SimCLR가 이 invariance를 학습할 수 있는지 봅니다.

**English**
Instead of images, we use 2D cluster data so we can visualise SimCLR's training dynamics quickly. We sample points from 4 Gaussian clusters and create positive pairs by adding small isotropic noise — two noisy copies of the same point share semantics (cluster) but differ in coordinates. We then test whether SimCLR can learn this invariance.

In [ ]:
def make_clusters(
    n_per_class: int = 256,
    n_classes: int = 4,
    radius: float = 4.0,
    spread: float = 0.6,
) -> tuple[np.ndarray, np.ndarray]:
    """Generate 2D Gaussian clusters arranged on a circle.

    Args:
        n_per_class: points per cluster.
        n_classes: number of clusters.
        radius: distance of cluster centers from origin.
        spread: standard deviation of points within a cluster.

    Returns:
        x: float32 array (n_per_class*n_classes, 2) of points.
        y: int64 array of cluster labels (used only for visualisation).
    """
    xs, ys = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes
        center = np.array([radius * np.cos(angle), radius * np.sin(angle)])
        pts = center + spread * np.random.randn(n_per_class, 2)
        xs.append(pts)
        ys.append(np.full(n_per_class, k, dtype=np.int64))
    return (
        np.concatenate(xs).astype(np.float32),
        np.concatenate(ys),
    )


class TwoViewDataset(Dataset):
    """Wraps 2D points and emits two augmented views per __getitem__ call.

    Augmentation = additive Gaussian noise. Two independent draws of noise create
    a positive pair; SimCLR is asked to make their embeddings agree.
    """

    def __init__(self, x: np.ndarray, noise_std: float = 0.4):
        self.x = torch.from_numpy(x)
        self.noise_std = noise_std

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        anchor = self.x[idx]
        view_a = anchor + self.noise_std * torch.randn_like(anchor)
        view_b = anchor + self.noise_std * torch.randn_like(anchor)
        return view_a, view_b


x_data, y_data = make_clusters(n_per_class=256, n_classes=4)
print(f'Data shape: {x_data.shape}, labels: {np.unique(y_data)}')

# Visualise the raw data
fig, ax = plt.subplots(figsize=(6, 6))
for k in range(4):
    pts = x_data[y_data == k]
    ax.scatter(pts[:, 0], pts[:, 1], s=8, label=f'class {k}')
ax.set_title('Synthetic 2D clusters / 합성 2D 클러스터')
ax.set_aspect('equal')
ax.legend()
plt.show()

## Part 3: Encoder + Projection Head / 인코더 + Projection 헤드

**한국어**
SimCLR Figure 2 구조를 그대로 따릅니다: 작은 base encoder $f$로 표현 $h$를 만들고, projection head $g$로 contrastive 공간 $z$로 매핑. 학습 후에는 $g$를 버리고 $h$만 downstream에 사용. 2D 데이터이므로 ResNet 대신 2-layer MLP encoder를 사용.

**English**
We follow SimCLR Figure 2 exactly: a small base encoder $f$ produces representation $h$; the projection head $g$ maps it to contrastive space $z$. After training, $g$ is discarded; only $h$ is used downstream. Since the data is 2D, we use a 2-layer MLP encoder instead of a ResNet.

In [ ]:
class Encoder(nn.Module):
    """Tiny base encoder f: maps 2D input to representation h in R^{repr_dim}."""

    def __init__(self, in_dim: int = 2, hidden_dim: int = 64, repr_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, repr_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ProjectionHead(nn.Module):
    """Nonlinear projection head g: R^{repr_dim} -> R^{proj_dim}.

    Same form as SimCLR (Eq. in §2.1): z = W2 * ReLU(W1 * h).
    """

    def __init__(self, repr_dim: int = 32, hidden_dim: int = 32, proj_dim: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(repr_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.net(h)


class SimCLRModel(nn.Module):
    """Full SimCLR model: encoder f then projection head g."""

    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.projector = ProjectionHead()

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.encoder(x)
        z = self.projector(h)
        return h, z


model = SimCLRModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'SimCLR mini-model parameter count: {n_params}')
print(model)

## Part 4: Training Loop / 학습 루프

**한국어**
Algorithm 1을 따라 매 미니배치에서: (1) augment 두 view, (2) encoder + projection으로 $z_a, z_b$ 계산, (3) NT-Xent 손실로 backprop, (4) 가중치 갱신. 학습 전후의 표현 변화를 비교하기 위해 학습 시작 전 $h$ snapshot을 따로 저장합니다.

**English**
Following Algorithm 1: for each mini-batch (1) augment two views, (2) compute $z_a, z_b$ via encoder + projection, (3) backprop NT-Xent, (4) update. We save a snapshot of $h$ before training to compare against post-training $h$.

In [ ]:
def encode_all(model: SimCLRModel, x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Run the encoder + projection on all points; return (h, z) numpy arrays."""
    model.eval()
    with torch.no_grad():
        x_t = torch.from_numpy(x).to(device)
        h, z = model(x_t)
    return h.cpu().numpy(), z.cpu().numpy()


# Snapshot h before any training (random init)
h_before, z_before = encode_all(model, x_data)

# Training
dataset = TwoViewDataset(x_data, noise_std=0.4)
loader = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
criterion = NTXentLoss(temperature=0.5)

n_epochs = 60
loss_history: list[float] = []
model.train()
for epoch in range(n_epochs):
    epoch_losses: list[float] = []
    for view_a, view_b in loader:
        view_a = view_a.to(device)
        view_b = view_b.to(device)
        _, z_a = model(view_a)
        _, z_b = model(view_b)
        loss = criterion(z_a, z_b)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    mean_loss = float(np.mean(epoch_losses))
    loss_history.append(mean_loss)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch + 1:3d} | NT-Xent loss = {mean_loss:.4f}')

h_after, z_after = encode_all(model, x_data)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history)
ax.set_xlabel('epoch')
ax.set_ylabel('mean NT-Xent loss')
ax.set_title('SimCLR training loss / SimCLR 학습 손실')
plt.show()

## Part 5: t-SNE Visualization of Learned Embeddings / 학습된 임베딩의 t-SNE 시각화

**한국어**
세 공간을 t-SNE로 시각화합니다: (1) 학습 *전* $h$ — 무작위 초기화 인코더의 출력, 클러스터가 뚜렷하지 않음, (2) 학습 *후* $h$ — projection head 직전의 표현으로 SimCLR 논문이 downstream에 권장, (3) 학습 *후* $z = g(h)$ — contrastive 공간. 클러스터 라벨은 학습 신호로 *전혀* 사용되지 않았음을 강조합니다 — 그럼에도 두 view의 일치만으로 의미적 구조가 emerge합니다.

**English**
We visualise three spaces with t-SNE: (1) $h$ *before* training — output of the random encoder, no clear cluster structure; (2) $h$ *after* training — pre-projection representation, recommended for downstream by the paper; (3) $z = g(h)$ *after* training — the contrastive space. Crucially, cluster labels were *never* used as supervision — semantic structure emerges purely from view-agreement.

In [ ]:
def tsne_2d(arr: np.ndarray, perplexity: float = 30.0, seed: int = 0) -> np.ndarray:
    """Project arr to 2D via t-SNE."""
    return TSNE(
        n_components=2,
        perplexity=perplexity,
        init='pca',
        learning_rate='auto',
        random_state=seed,
    ).fit_transform(arr)


h_before_2d = tsne_2d(h_before)
h_after_2d = tsne_2d(h_after)
z_after_2d = tsne_2d(z_after)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, emb_2d, title in zip(
    axes,
    [h_before_2d, h_after_2d, z_after_2d],
    [
        'h before training / 학습 전 h',
        'h after training / 학습 후 h',
        'z = g(h) after training / 학습 후 z',
    ],
):
    for k in range(4):
        pts = emb_2d[y_data == k]
        ax.scatter(pts[:, 0], pts[:, 1], s=8, label=f'class {k}')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()


# Quantitative check: simple kNN classification accuracy on h before vs after
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split


def knn_accuracy(features: np.ndarray, labels: np.ndarray, k: int = 5) -> float:
    """Train/test split + kNN classification accuracy as a representation-quality probe."""
    f_tr, f_te, y_tr, y_te = train_test_split(
        features, labels, test_size=0.3, random_state=0, stratify=labels
    )
    clf = KNeighborsClassifier(n_neighbors=k).fit(f_tr, y_tr)
    return float(clf.score(f_te, y_te))


acc_h_before = knn_accuracy(h_before, y_data)
acc_h_after = knn_accuracy(h_after, y_data)
acc_z_after = knn_accuracy(z_after, y_data)
print(f'kNN accuracy on h (before training): {acc_h_before:.3f}')
print(f'kNN accuracy on h (after training):  {acc_h_after:.3f}')
print(f'kNN accuracy on z (after training):  {acc_z_after:.3f}')
print('\nSimCLR paper finding: h tends to outperform z for downstream tasks (>10% gap on ImageNet).')

## Summary / 요약

| Concept / 개념 | This Notebook / 본 노트북 | SimCLR Paper / 논문 원본 |
|---|---|---|
| Augmentation / 증강 | Additive Gaussian noise on 2D points / 2D 점에 가우시안 노이즈 | Random crop + color distortion + Gaussian blur on ImageNet 이미지 |
| Base encoder $f$ / 기본 인코더 | 2-layer MLP (2 → 64 → 32) | ResNet-50 / ResNet-50(2×) / ResNet-50(4×) |
| Projection head $g$ / Projection 헤드 | 2-layer MLP (32 → 32 → 16) | 2-layer MLP (2048 → 2048 → 128) |
| Loss / 손실 | NT-Xent (Eq. 1), $\tau$=0.5 | NT-Xent (Eq. 1), $\tau$=0.1–0.5 |
| Batch size / 배치 크기 | 128 | 256–8192 (default 4096) |
| Optimizer / 최적화기 | Adam | LARS (You 2017) |
| Epochs / 에폭 | 60 (synthetic) | 100–1000 (ImageNet) |
| Linear eval / 선형 평가 | kNN accuracy as proxy / 대용으로 kNN 정확도 | Linear classifier on frozen $f$, ImageNet top-1/top-5 |

**한국어**
- NT-Xent 손실은 cosine similarity + temperature + cross-entropy의 조합으로, 단 몇 줄로 깔끔히 구현됩니다.
- 데이터 라벨을 *전혀* 사용하지 않고 "같은 점에서 나온 두 view의 일치"라는 신호만으로 클래스 구조가 emerge합니다.
- 논문의 핵심 통찰 "$h$ > $z$ for downstream"은 이 미니 데모에서도 약하게 관찰될 수 있습니다 (kNN 정확도 비교).
- 실제 SimCLR는 큰 batch + 강한 augmentation + LARS + 1000 epoch + ResNet-50(4×)로 ImageNet 76.5% top-1을 달성합니다.

**English**
- NT-Xent reduces to cosine similarity + temperature + cross-entropy, implementable in a handful of lines.
- Without any label supervision, semantic class structure emerges from "agreement of two views of the same point".
- The paper's key insight "$h$ > $z$ downstream" can be weakly reproduced here via the kNN-accuracy probe.
- The full SimCLR — large batch + strong augmentation + LARS + 1000 epochs + ResNet-50(4×) — reaches 76.5% top-1 on ImageNet.